# Derivation: Single Index Model Covariance Matrix

In this derivation, we show how to compute the covariance between two assets using the single index model. This is a key result for portfolio optimization, as it allows us to build covariance matrices without estimating all pairwise correlations directly from data.

> __Learning Objectives:__
>
> By the end of this notebook, we will be able to:
> * __SIM covariance structure:__ We will derive the pairwise covariance of growth rates under the SIM from first principles. We will see why all inter-asset correlation flows through the common market factor.
> * __Systematic vs idiosyncratic decomposition:__ We will decompose total variance into a rank-1 systematic term and a diagonal idiosyncratic term. We will count parameters to see why SIM stabilizes portfolio optimization.
> * __Storage convention for $\sigma^2_{\varepsilon}$:__ We will recognize that $\Delta{t}\,\sigma^2_{\varepsilon}$ on the diagonal is a storage-unit choice, not a dimensional conversion. We will be able to check unit consistency in downstream code.

___

## The Single Index Model

Recall the single index model for firm $i$ at time $t$:
$$
g^{(t)}_{i} = \alpha_{i} + \beta_{i}\;g^{(t)}_{m} + \varepsilon^{(t)}_{i}
$$
where $\alpha_{i}$ is the idiosyncratic (firm-specific) growth, $\beta_{i}$ is the sensitivity of firm $i$ to market movements, $g^{(t)}_{m}$ is the growth rate of the market index at time $t$, and $\varepsilon^{(t)}_{i}$ is the error term for firm $i$ (describes growth rate not captured by the firm or market factors).

> __Error Model Assumptions__
>
> We assume the error model has the following properties:
> $$
\begin{align*}
\mathbb{E}[\varepsilon^{(t)}_{i}] &= 0\\
\text{Var}(\varepsilon^{(t)}_{i}) &= \Delta{t}\;\sigma^2_{\varepsilon_i}\\
\text{Cov}(\varepsilon^{(t)}_{i}, g^{(t)}_{m}) &= 0\\
\text{Cov}(\varepsilon^{(t)}_{i}, \varepsilon^{(t)}_{j}) &= 0\quad\text{for }i \neq j
\end{align*}
$$

The first assumption says errors have zero mean. The second specifies the error variance, where $\Delta{t}$ is the time step (e.g., $\Delta{t} = 1/252$ for daily data annualized). The third assumption says errors are uncorrelated with the market. The fourth and most crucial assumption says that after accounting for market movements, the remaining variation in firm returns is independent across firms. This is what allows us to dramatically simplify the covariance structure.

___

## Derivation: Covariance Between Two Different Assets

Let's compute the covariance between the growth rates of two different firms $i$ and $j$ where $i \neq j$. Starting with the definition of covariance:
$$
\text{Cov}(g_i, g_j) = \mathbb{E}\left[(g_i - \mathbb{E}[g_i])(g_j - \mathbb{E}[g_j])\right]
$$

First, we need the expected growth rates. Taking expectations of the SIM yields:
$$
\begin{align*}
\mathbb{E}[g_i] &= \mathbb{E}[\alpha_{i} + \beta_{i}\;g_{m} + \varepsilon_{i}]\\
&= \alpha_{i} + \beta_{i}\;\mathbb{E}[g_{m}] + \mathbb{E}[\varepsilon_{i}]\\
&= \alpha_{i} + \beta_{i}\;\mathbb{E}[g_{m}]
\end{align*}
$$
Similarly, $\mathbb{E}[g_j] = \alpha_{j} + \beta_{j}\;\mathbb{E}[g_{m}]$.

The deviations from the mean are given by:
$$
\begin{align*}
g_i - \mathbb{E}[g_i] &= (\alpha_{i} + \beta_{i}\;g_{m} + \varepsilon_{i}) - (\alpha_{i} + \beta_{i}\;\mathbb{E}[g_{m}])\\
&= \beta_{i}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{i}
\end{align*}
$$
Similarly, $g_j - \mathbb{E}[g_j] = \beta_{j}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{j}$.

Substituting into the covariance formula yields:
$$
\begin{align*}
\text{Cov}(g_i, g_j) &= \mathbb{E}\left[\left(\beta_{i}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{i}\right)\left(\beta_{j}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{j}\right)\right]
\end{align*}
$$

Expanding the product gives:
$$
\begin{align*}
\text{Cov}(g_i, g_j) &= \mathbb{E}\Big[\beta_{i}\beta_{j}(g_{m} - \mathbb{E}[g_{m}])^2 + \beta_{i}(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{j}\\
&\quad + \beta_{j}(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i} + \varepsilon_{i}\varepsilon_{j}\Big]
\end{align*}
$$

Taking expectations term by term using linearity yields:
$$
\begin{align*}
\text{Cov}(g_i, g_j) &= \beta_{i}\beta_{j}\;\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])^2] + \beta_{i}\;\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{j}]\\
&\quad + \beta_{j}\;\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i}] + \mathbb{E}[\varepsilon_{i}\varepsilon_{j}]
\end{align*}
$$

Now we apply our assumptions:
* $\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])^2] = \text{Var}(g_m) = \sigma^2_m$ (market variance)
* $\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{j}] = \text{Cov}(g_m, \varepsilon_j) = 0$ (market uncorrelated with errors)
* $\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i}] = \text{Cov}(g_m, \varepsilon_i) = 0$ (market uncorrelated with errors)
* $\mathbb{E}[\varepsilon_{i}\varepsilon_{j}] = \text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$ (errors uncorrelated across firms)

Substituting the assumptions back into the expansion, we obtain:
$$
\boxed{\text{Cov}(g_i, g_j) = \beta_{i}\beta_{j}\;\sigma^2_m \quad \text{for } i \neq j\quad\blacksquare}
$$

> __Key Insight:__
>
> The covariance between any two different assets depends only on their market exposures ($\beta_i$ and $\beta_j$) and the market variance ($\sigma^2_m$). All correlation between assets arises through their common exposure to market movements.

___

## Derivation: Variance of a Single Asset

Now let's compute the variance of a single asset $i$, which is the special case where $i = j$ in the covariance formula. The variance is given by:
$$
\text{Var}(g_i) = \text{Cov}(g_i, g_i) = \mathbb{E}\left[(g_i - \mathbb{E}[g_i])^2\right]
$$

From our earlier work, we have:
$$
g_i - \mathbb{E}[g_i] = \beta_{i}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{i}
$$

Squaring both sides yields:
$$
\begin{align*}
(g_i - \mathbb{E}[g_i])^2 &= \left(\beta_{i}(g_{m} - \mathbb{E}[g_{m}]) + \varepsilon_{i}\right)^2\\
&= \beta^2_{i}(g_{m} - \mathbb{E}[g_{m}])^2 + 2\beta_{i}(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i} + \varepsilon^2_{i}
\end{align*}
$$

Taking expectations gives:
$$
\begin{align*}
\text{Var}(g_i) &= \beta^2_{i}\;\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])^2] + 2\beta_{i}\;\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i}] + \mathbb{E}[\varepsilon^2_{i}]
\end{align*}
$$

Applying our assumptions where $\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])^2] = \sigma^2_m$ is the market variance, $\mathbb{E}[(g_{m} - \mathbb{E}[g_{m}])\varepsilon_{i}] = 0$ because the market is uncorrelated with errors, and $\mathbb{E}[\varepsilon^2_{i}] = \text{Var}(\varepsilon_i) = \Delta{t}\;\sigma^2_{\varepsilon_i}$ is the error variance, we obtain:
$$
\boxed{\text{Var}(g_i) = \beta^2_{i}\;\sigma^2_m + \Delta{t}\;\sigma^2_{\varepsilon_i}\quad\blacksquare}
$$

> __Risk Decomposition:__
>
> The total variance of firm $i$ consists of two components: __systematic risk__ $\beta^2_{i}\;\sigma^2_m$, which is the risk from market exposure that cannot be diversified away, and __idiosyncratic risk__ $\Delta{t}\;\sigma^2_{\varepsilon_i}$, which is firm-specific risk that can be diversified away in a portfolio.

___

## The Complete SIM Covariance Formula

Combining our results, we have the general covariance formula:
$$
\boxed{
\text{Cov}(g_i, g_j) = \begin{cases}
\beta^2_{i}\;\sigma^2_m + \Delta{t}\;\sigma^2_{\varepsilon_i} & \text{if } i = j\\
\beta_{i}\beta_{j}\;\sigma^2_m & \text{if } i \neq j
\end{cases}\quad\blacksquare
}
$$

This formula is the foundation for constructing covariance matrices in SIM-based portfolio optimization.

### Constructing the Covariance Matrix

For a portfolio with $N$ assets, the covariance matrix $\boldsymbol{\Sigma}$ is an $N \times N$ symmetric matrix where:
$$
\boldsymbol{\Sigma}_{i,j} = \text{Cov}(g_i, g_j)
$$

Using the SIM, we can construct this matrix element by element:

__Diagonal elements__ (variances):
$$
\boldsymbol{\Sigma}_{i,i} = \beta^2_{i}\;\sigma^2_m + \Delta{t}\;\sigma^2_{\varepsilon_i}
$$

__Off-diagonal elements__ (covariances):
$$
\boldsymbol{\Sigma}_{i,j} = \beta_{i}\beta_{j}\;\sigma^2_m \quad \text{for } i \neq j
$$

> __Computational Advantage__ 
>
> Using the SIM, we only need to estimate $N$ beta values ($\beta_1, \beta_2, \ldots, \beta_N$), $N$ residual variances ($\sigma^2_{\varepsilon_1}, \sigma^2_{\varepsilon_2}, \ldots, \sigma^2_{\varepsilon_N}$), and one market variance ($\sigma^2_m$). This is a total of $2N + 1$ parameters, compared to $N(N+1)/2$ unique elements in a full covariance matrix. For a 100-asset portfolio, this reduces from 5,050 parameters to just 201!

___

## Example: Three-Asset Portfolio
Let's construct the covariance matrix for a simple three-asset portfolio to see the construction in practice. All rates are in units of 1/year (annualized), and $\Delta{t} = 1/252$ for daily sampling. Suppose we have estimated the following SIM parameters:

* Asset 1: $\beta_1 = 0.8$, $\sigma^2_{\varepsilon_1} = 0.04\;\text{yr}^{-1}$
* Asset 2: $\beta_2 = 1.2$, $\sigma^2_{\varepsilon_2} = 0.06\;\text{yr}^{-1}$
* Asset 3: $\beta_3 = 1.5$, $\sigma^2_{\varepsilon_3} = 0.09\;\text{yr}^{-1}$
* Market: $\sigma^2_m = 0.04\;\text{yr}^{-2}$

> __Unit check:__
>
> In the course storage convention, $\sigma^2_{\varepsilon_i}$ carries units of inverse time (1/yr), and $\Delta{t}\,\sigma^2_{\varepsilon_i}$ recovers the per-step residual variance in $\text{yr}^{-2}$, consistent with the $\beta^2_i\,\sigma^2_m$ systematic term. See also the note in the companion deeper-dive notebook.

### Diagonal Elements (variances)
The diagonal entries are given by:
$$
\begin{align*}
\boldsymbol{\Sigma}_{1,1} &= (0.8)^2(0.04) + \tfrac{1}{252}(0.04) = 0.0256 + 0.000159 = 0.02576\\
\boldsymbol{\Sigma}_{2,2} &= (1.2)^2(0.04) + \tfrac{1}{252}(0.06) = 0.0576 + 0.000238 = 0.05784\\
\boldsymbol{\Sigma}_{3,3} &= (1.5)^2(0.04) + \tfrac{1}{252}(0.09) = 0.0900 + 0.000357 = 0.09036
\end{align*}
$$

### Off-Diagonal Elements (covariances)
The off-diagonal entries are given by:
$$
\begin{align*}
\boldsymbol{\Sigma}_{1,2} = \boldsymbol{\Sigma}_{2,1} &= (0.8)(1.2)(0.04) = 0.0384\\
\boldsymbol{\Sigma}_{1,3} = \boldsymbol{\Sigma}_{3,1} &= (0.8)(1.5)(0.04) = 0.0480\\
\boldsymbol{\Sigma}_{2,3} = \boldsymbol{\Sigma}_{3,2} &= (1.2)(1.5)(0.04) = 0.0720
\end{align*}
$$

### The Complete Covariance Matrix
Assembling the diagonal and off-diagonal entries, the full covariance matrix is given by:
$$
\boldsymbol{\Sigma} = \begin{bmatrix}
0.02576 & 0.0384  & 0.0480\\
0.0384  & 0.05784 & 0.0720\\
0.0480  & 0.0720  & 0.09036
\end{bmatrix}\;\text{yr}^{-2}
$$

> __Observation:__
>
> The strongest covariance (0.0720) is between assets 2 and 3, which both have high betas. High-beta assets co-move because both amplify market movements. The idiosyncratic contribution $\Delta{t}\,\sigma^2_{\varepsilon_i}$ on the diagonal is roughly two to three orders of magnitude smaller than the systematic $\beta^2_i\,\sigma^2_m$ term under daily sampling: this is the $\Delta{t}$ storage convention at work, not a claim that idiosyncratic risk is negligible in absolute terms.

___

## Summary

In this derivation, we showed how the single index model dramatically simplifies the computation of asset covariances. Starting with the SIM for each firm, we computed the covariance between two firms using the definition and properties of expectation. By applying the key assumption that error terms are uncorrelated across firms and with the market, we obtained a general result that depends only on betas and market variance.

> __Key Results__
>
> The covariance between two different assets is $\text{Cov}(g_i, g_j) = \beta_i\beta_j\;\sigma^2_m$ for $i \neq j$, meaning all correlations arise from common market exposure. The variance of a single asset is $\text{Var}(g_i) = \beta^2_i\;\sigma^2_m + \Delta{t}\;\sigma^2_{\varepsilon_i}$, decomposing total risk into systematic and idiosyncratic components. This dramatically simplifies portfolio optimization by reducing the number of parameters from $O(N^2)$ to $O(N)$.

> __Key Takeaways:__
>
> * __Pairwise covariance is rank-1 in $\boldsymbol{\beta}$:__ Off-diagonal covariances depend only on $\beta_i\beta_j\sigma^2_m$, so correlation structure is fully determined by market exposure. This lets the solver work with $2N+1$ parameters instead of $N(N+1)/2$.
> * __Diagonal variance adds idiosyncratic risk:__ The diagonal gains a $\Delta{t}\,\sigma^2_{\varepsilon_i}$ term that captures firm-specific risk. Diversifying across many low-correlation names shrinks the portfolio-level contribution of this term.
> * __Units matter:__ Both $\beta^2_i\sigma^2_m$ and $\Delta{t}\,\sigma^2_{\varepsilon_i}$ must be expressed in the same time-scale. Mixing conventions is the single most common source of bugs in SIM covariance code; the companion deeper-dive notebook walks through the course-wide convention.

___